# Part 2: Time series analysis

Owner: _(name)_

**What this notebook is for.** Describe how the series behaves before any model sees it: does it
trend, does it repeat in a cycle, does it stay around one level, does the past say anything about the
future. Everything here uses the **training period only**, so the test period stays unseen.

**What you do.** Run the cells in order. Read the short note above each cell: it says what to look for
in the output. Section 0 decides which years we train on, the question the professor asked us to
justify. At the end, write the decision section.

**What you hand over.** This notebook with outputs, the text for section 2 of the report (brief: trend,
seasonality, stationarity tests with their assumptions and limits, autocorrelation, the random walk),
and the period decision (`TRAIN_START` in `src/config.py`, with the evidence from section 0).

### Setup: run this cell first (nothing to change)

It fetches the repository and the data into this Colab session, loads the shared settings, and prints the
three periods: train, validation, test. The first run takes about a minute.

In [ ]:
# ---- Setup (identical in every notebook). Just run it. ----
import sys, os, pathlib, subprocess, importlib.util

REPO_URL = "https://github.com/Makkaii7/sp500-forecasting.git"
EXTRA_PACKAGES = []      # what this notebook needs that Colab does not have by default

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 1. In Colab, fetch the repository (once per session). On your own computer this step is skipped.
if IN_COLAB and not pathlib.Path("sp500-forecasting").exists() and not pathlib.Path("src/config.py").exists():
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    public = subprocess.run(["git", "clone", "--quiet", REPO_URL], env=env, capture_output=True).returncode == 0
    if not public:                                   # only if the repository was made private again
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            from getpass import getpass
            token = getpass("The repository is private. Paste a GitHub token: ")
        subprocess.run(["git", "clone", "--quiet", REPO_URL.replace("https://", f"https://{token}@")], env=env, check=True)
if IN_COLAB and pathlib.Path("sp500-forecasting").exists():
    os.chdir("sp500-forecasting")

# 2. Install anything missing (usually only on the first run in Colab).
missing = [p for p in EXTRA_PACKAGES if importlib.util.find_spec(p) is None]
if missing:
    print("installing", ", ".join(missing), "... (about a minute)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

# 3. Find the repository folder and load the shared code (src/config.py and src/common.py).
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config, common

# 4. The data everyone uses, and the three periods in time order.
df = common.load_prices()           # columns: Open, High, Low, Close, Volume, log_return; one row per trading day
y = common.target_series(df)        # the daily log return: what the models work on
train, val, test = common.split(y)  # dates from src/config.py
print("repository:", root)
print("data:", df.index.min().date(), "to", df.index.max().date(), "|", len(df), "trading days")
print(f"train       {train.index.min().date()} to {train.index.max().date()}  ({len(train)} days)  the model learns here")
print(f"validation  {val.index.min().date()} to {val.index.max().date()}  ({len(val)} days)  you compare settings here")
print(f"test        {test.index.min().date()} to {test.index.max().date()}  ({len(test)} days)  touched once, at the very end")
df.tail(3)

## 0. Which period do we train on? Evidence from the full history (just run, then decide)

The data file holds the S&P 500 from 1927. The professor asked for a justification of the period we
train on, not a preference. Three pieces of evidence, all computed on data **before the validation
period** so that nothing from validation or test leaks in:

1. **How the series behaved across history.** Volatility is the usual size of a daily move (the standard
   deviation of the daily return). A stretch where the series behaves differently from today is a
   *regime*; a date where the behaviour changes is a *structural break*. One plot and one table by decade.
2. **Does older data help a model?** For each candidate start year, the same simple model is fitted on
   the returns from that start to TRAIN_END and scored one day ahead on the validation year, in index
   points, next to the baseline. Only the start date changes.
3. **Scaling.** The neural networks squeeze their inputs into a fixed range using the training period's
   most extreme days. A 1929 or 1987 day stretches that range and squashes the modern years into a sliver.

Then write the decision in 0.5 and put it in `src/config.py` as `TRAIN_START`.

### 0.1 Volatility across history

In [ ]:
full = df.loc[: config.TRAIN_END]                        # everything before validation and test
rf = full["log_return"].dropna()
recent = rf[rf.index >= pd.Timestamp(config.TRAIN_END) - pd.DateOffset(years=10)]   # the last 10 years before validation
fig, ax = plt.subplots(2, 1, figsize=(11, 6.5))
full["Close"].plot(ax=ax[0], logy=True, title="S&P 500 close since 1927 (log scale: equal slopes mean equal % growth)")
ax[0].set_ylabel("index points")
(rf.rolling(252).std() * 100).plot(ax=ax[1], title="Rolling 1-year standard deviation of the daily return, in % (the usual size of a daily move)")
ax[1].axhline(recent.std() * 100, color="grey", ls="--", label="last 10 years before validation"); ax[1].legend()
plt.tight_layout()

### 0.2 The same thing as a table, by decade

In [ ]:
decade = (rf.index.year // 10) * 10
table = rf.groupby(decade).agg(days="count", mean_pct=lambda s: s.mean() * 100, std_pct=lambda s: s.std() * 100,
                               worst_day_pct=lambda s: s.min() * 100, best_day_pct=lambda s: s.max() * 100)
table["std vs last 10 years"] = table["std_pct"] / (recent.std() * 100)    # 1.0 = same size of moves as today
table.round(3)

### 0.3 Does older data help? Candidate start years on the validation year

The model is ARMA(1, 1) on the daily return (one past value, one past error), fitted once on each
candidate period with its parameters then frozen, and asked for one-day-ahead forecasts through the
validation year using only the days before each forecast. Errors in index points, next to the baseline.
Add or change candidates freely. If the column is flat, that is the finding: the start date barely
changes a one-day forecast, and the decision has to rest on 0.1, 0.2 and 0.4.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
CANDIDATES = ["1928-01-01", "1950-01-01", "1980-01-01", "2000-01-01", "2010-01-01"]
actual_val = common.actual_price(val.index, df)
baseline_val = common.rmse(actual_val, common.last_value_baseline(val.index, df))
rows = []
for start in CANDIDATES:
    tr = y.loc[start: config.TRAIN_END]
    fitted = ARIMA(tr.to_numpy(), order=(1, 0, 1)).fit()
    extended = fitted.extend(val.to_numpy())                                     # same parameters, new days
    pred = pd.Series(np.asarray(extended.get_prediction().predicted_mean), index=val.index)   # one step ahead, past only
    rmse = common.rmse(actual_val, common.to_price(pred, df))
    rows.append({"start": start, "training days": len(tr), "val RMSE (index points)": rmse, "vs baseline (%)": (rmse / baseline_val - 1) * 100})
print("baseline (tomorrow = today) val RMSE:", round(baseline_val, 2), "index points")
pd.DataFrame(rows).set_index("start").round(3)

### 0.4 Scaling: how much of the input range would the modern years use?

In [ ]:
rows = []
for start in CANDIDATES:
    tr = y.loc[start: config.TRAIN_END]
    rows.append({"start": start, "worst day %": tr.min() * 100, "best day %": tr.max() * 100,
                 "share of the scaled range used by the last 10 years": (recent.max() - recent.min()) / (tr.max() - tr.min())})
pd.DataFrame(rows).set_index("start").round(3)

### 0.5 Decision (write here)

Three sentences, each pointing at a number above: which decades behave like today and which do not
(0.1, 0.2); what the validation table said (0.3); what scaling would do with the extreme years (0.4).
Then: **TRAIN_START = ...**, put it in `src/config.py`, and tell the group.

_your text_

## From here on: the training period only

Everything below uses the period from `TRAIN_START` to `TRAIN_END`, so that nothing from validation or
test leaks into the analysis.

In [ ]:
train_df = df.loc[config.TRAIN_START: config.TRAIN_END]   # the training period only, nothing later
close = train_df["Close"]                        # the price level, in index points
r = train_df["log_return"].dropna()              # the daily log return
print("training period:", close.index.min().date(), "to", close.index.max().date(), "|", len(close), "days")

## 1. Trend: the price level against the daily return (just run)

A trend is a long-run direction. The orange line is a rolling mean: the average of the last 250 trading
days (about one year), recomputed every day, so it smooths out the wiggles. What to look for: the close
drifts upward over the years; the return wobbles around zero with no direction.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6))
close.plot(ax=ax[0], title="Training period close: a long-term upward direction (trend)")
close.rolling(250).mean().plot(ax=ax[0], label="250-day rolling mean"); ax[0].legend(); ax[0].set_ylabel("index points")
r.plot(ax=ax[1], title="Daily log return: wobbles around zero, no trend")
r.rolling(250).mean().plot(ax=ax[1], label="250-day rolling mean"); ax[1].legend()
plt.tight_layout()

## 2. Does the size of the daily moves change over time? (just run)

The standard deviation is the usual size of a daily move. Computed over a sliding 60-day window it shows
how that size changes. What to look for: calm stretches and stormy stretches that cluster together
(2020, for example). This matters because a model that assumes one constant spread is simplifying.

In [ ]:
r.rolling(60).std().plot(figsize=(11, 3), title="60-day rolling standard deviation of the daily return (the size of the moves)")

## 3. Seasonality: does anything repeat on a fixed cycle? (just run)

Seasonality means a pattern that comes back at a fixed, known interval. For daily trading data the
candidates are the week (5 trading days), the month (about 21) and the year (about 252). Three ways to
look, in the three cells below:

1. calendar tables: the average return by weekday and by month;
2. the autocorrelation at the seasonal lags (explained in section 5: how much a day resembles the day
   one week, one month or one year earlier);
3. a decomposition, which splits the series into trend, a repeating seasonal part and the rest.

Report what you see either way. "No seasonality worth modelling" is a finding, and the brief expects it
to be stated with evidence.

In [ ]:
by_weekday = r.groupby(r.index.dayofweek).agg(["mean", "std", "count"])
by_weekday.index = ["Mon", "Tue", "Wed", "Thu", "Fri"]
by_month = r.groupby(r.index.month).agg(["mean", "std", "count"])
display(by_weekday.round(5)); display(by_month.round(5))

In [ ]:
from statsmodels.tsa.stattools import acf
lags = [1, 2, 3, 5, 10, 21, 63, 126, 252]                 # 5 = a week, 21 = a month, 252 = a year
vals = acf(r, nlags=max(lags), fft=True)
limit = 1.96 / np.sqrt(len(r))                            # anything inside this band is too small to trust
pd.DataFrame({"lag": lags, "ACF": [round(vals[k], 4) for k in lags], "beyond the band": [abs(vals[k]) > limit for k in lags]})

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
dec = seasonal_decompose(close, model="multiplicative", period=252)   # yearly cycle, as an illustration
fig = dec.plot(); fig.set_size_inches(11, 8); plt.tight_layout()

## 4. Stationarity tests (just run)

A series is **stationary** when it wobbles around a fixed level with a steady spread, so that what a model
learns from the past keeps applying. The price is not stationary (it drifts); the daily return roughly is.
ARIMA and the neural networks want a stationary input, which is why the models work on the return.

Two tests with opposite starting points, so read them together:

- **ADF** (augmented Dickey-Fuller). Starting assumption: *not* stationary. A small p-value (under 0.05)
  says "reject that": evidence **for** stationarity. Limits: weak when the series is almost but not quite
  a random walk, and the answer depends on how many lags it includes.
- **KPSS**. Starting assumption: stationary. A small p-value says "reject that": evidence **against**
  stationarity. Limits: its p-values come from a table and stop at 0.01 and 0.10.

What to look for: Close gives ADF large and KPSS small (not stationary); log return gives ADF small and
KPSS large (stationary).

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
import warnings

def stationarity_report(series, name):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")          # KPSS warns when the p-value hits the table limits
        adf_stat, adf_p, *_ = adfuller(series.dropna(), autolag="AIC")
        kpss_stat, kpss_p, *_ = kpss(series.dropna(), regression="c", nlags="auto")
    verdict = "stationary" if (adf_p < 0.05 and kpss_p >= 0.05) else "NOT stationary" if (adf_p >= 0.05 and kpss_p < 0.05) else "mixed, look at the plots"
    print(f"{name:12s} ADF stat = {adf_stat:7.3f} p = {adf_p:.4f}   KPSS stat = {kpss_stat:6.3f} p = {kpss_p:.4f}   -> {verdict}")

stationarity_report(close, "Close")
stationarity_report(r, "log return")

## 5. ACF and PACF: how far back does the past matter? (just run)

The **autocorrelation** at lag k is the correlation between today's value and the value k days earlier
(lag 1 = yesterday, lag 2 = the day before, and so on). The ACF plot shows it for many lags at once. The
**PACF** shows the same thing after removing what the shorter lags already explain. Bars inside the shaded
band are too small to trust.

What to look for: the close's ACF stays near 1 for many lags (yesterday's price almost equals today's,
a sign of non-stationarity). The return's bars sit mostly inside the band: knowing yesterday's return
tells you little about today's.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
plot_acf(close, lags=40, ax=ax[0, 0], title="ACF of Close (fades slowly: not stationary)")
plot_pacf(close, lags=40, ax=ax[0, 1], title="PACF of Close")
plot_acf(r, lags=40, ax=ax[1, 0], title="ACF of log return (inside the band: close to white noise)")
plot_pacf(r, lags=40, ax=ax[1, 1], title="PACF of log return")
plt.tight_layout()

## 6. The random walk, formally (read, then run)

A random walk says: tomorrow's value is today's value plus a surprise.

$$y_t = y_{t-1} + \varepsilon_t \qquad (\text{with drift: } y_t = c + y_{t-1} + \varepsilon_t)$$

The surprises $\varepsilon_t$ are called innovations. The assumption is that they are **white noise**:
average zero, constant spread, and unrelated to each other and to the past. In ARIMA notation this is
ARIMA(0, 1, 0). If the index really is a random walk, the best possible forecast of tomorrow's close is
today's close (plus the drift), which is exactly the baseline every model in this report must beat.

Evidence to check: the price fails the stationarity tests, its first difference (the return) passes them,
and the return's autocorrelations are all close to zero. The Ljung-Box test below checks that last point:
its starting assumption is that the first k autocorrelations are all zero; a small p-value says some of
them are not.

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox
lb = acorr_ljungbox(r, lags=[5, 10, 20], return_df=True)
lb["verdict"] = np.where(lb["lb_pvalue"] < 0.05, "some autocorrelation", "consistent with white noise")
lb

## 7. Decision (write here)

Write the verdict in plain words. Replace the examples with what the cells above showed.

- Stationarity: the price level is not stationary (ADF p = ..., KPSS p = ..., ACF fades slowly). One
  difference of the log price, the daily log return, is stationary (ADF p = ..., KPSS p = ...). So the
  models work on the return: d = 1 on the price, which is the same as d = 0 on the return.
- Seasonality: (what the weekday and month tables, the seasonal-lag ACF and the decomposition showed,
  and whether anything is worth modelling).
- Random walk: (whether the evidence fits it: innovations near white noise or not, and any small
  autocorrelation found at short lags).
- The split in `src/config.py`: TRAIN_START = ... (from section 0), validation = the 12 months after
  TRAIN_END, test = the 12 months after VAL_END, all in time order.